# Appendix — Self-tests & Sanity Checks (Human, Mouse, Pig)

*Last updated:* 2026-11-12

Use this notebook when you want to **confirm your setup is correct** after building graphs.

It checks:
- external YAML files exist
- graphs can be built/loaded
- a few representative conversions work
- (optional) deeper integrity checks via `TrackTests`

> **Tip:** If you hit an error, cross-reference Part 7.3 (`07_advanced_topics.ipynb`) for a diagnostic checklist.


## 1. Setup


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack
from idtrack import DB

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)


Local repository: idtrack_cache


## 2. Helper: pick a representative gene ID from a built graph

We avoid hard-coding gene IDs for mouse/pig by selecting an example **base Ensembl gene ID** directly from
the graph (these look like `ENSG...`, `ENSMUSG...`, `ENSSSCG...`; the macaques use `ENSMMUG...` and `ENSMFAG...`).


In [2]:
def pick_example_base_gene_id(graph) -> str:
    # The first base Ensembl gene id in the order of the node names (deterministic).
    for node in graph.nodes:
        if graph.node_type(node) == DB.nts_base_ensembl['gene']:
            return str(node)
    raise RuntimeError('No base_ensembl_gene node found in the graph.')

## 3. The checklist (repeat per organism)

For each organism we will:
1. confirm the YAML exists (or explain defaults)
2. build/load the graph snapshot
3. run a small conversion smoke test


### 3A. Human

In [3]:
# 3A.1 YAML presence
human_yaml = LOCAL_REPOSITORY / 'homo_sapiens_externals_modified.yml'
print('OK' if human_yaml.exists() else 'NOTE: not in the local repository (IDTrack uses the packaged default for human)', human_yaml)


OK idtrack_cache/homo_sapiens_externals_modified.yml


In [4]:
# 3A.2 Build/load graph
organism, latest_release = api.resolve_organism('human')
SNAPSHOT_RELEASE = 115  # pinned (latest_release is Ensembl's newest)
api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE)

g = api.track.graph
print('Organism:', g.graph['organism'])
print('Release:', g.graph['ensembl_release'])
print('Assembly:', g.graph['genome_assembly'])


2026-11-12 23:11:50 INFO:idtrack._verify_organism: Ensembl Rest API query to get the organism names and associated releases.


Organism: homo_sapiens
Release: 115
Assembly: 38


In [5]:
# 3A.3 Smoke conversion
example_id = pick_example_base_gene_id(api.track.graph)
result = api.convert_identifier(example_id, from_release=SNAPSHOT_RELEASE, to_release=SNAPSHOT_RELEASE)
result


{'query_id': 'ENSG00000000003',
 'graph_id': 'ENSG00000000003',
 'match': 'exact',
 'known_versions': [],
 'from_release': 115,
 'from_release_inferred': False,
 'from_release_given': 115,
 'from_release_fallback': False,
 'from_databases': ['base_ensembl_gene'],
 'to_release': 115,
 'strategy': 'best',
 'final_database': 'ensembl_gene',
 'target_id': ['ENSG00000000003.17'],
 'last_node': [['ENSG00000000003.17', 'ENSG00000000003.17']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': [],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

Interpretation tips:
- `no_corresponding=True` means the input ID didn't match anything in the graph.
- `no_conversion=True` means it matched, but IDTrack could not find a path to the target release.
- `target_id` is a list because ambiguity is possible (1→n).


### 3B. Mouse

In [6]:
mouse_yaml = LOCAL_REPOSITORY / 'mus_musculus_externals_modified.yml'
# A local file (Part 2) overrides the packaged default; either is fine.
shipped_mouse_yaml = Path(idtrack.__file__).parent / 'default_config' / mouse_yaml.name
HAS_MOUSE_YAML = mouse_yaml.exists() or shipped_mouse_yaml.exists()
if mouse_yaml.exists():
    print('OK, local file:', mouse_yaml)
else:
    print('OK, packaged default:' if HAS_MOUSE_YAML else 'MISSING (create it in Part 2 for mouse):', shipped_mouse_yaml.name)

OK, local file: idtrack_cache/mus_musculus_externals_modified.yml


In [7]:
if HAS_MOUSE_YAML:
    organism, latest_release = api.resolve_organism('mus musculus')
    SNAPSHOT_RELEASE = 115  # pinned (latest_release is Ensembl's newest)
    api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE)

    g = api.track.graph
    print('Organism:', g.graph['organism'])
    print('Release:', g.graph['ensembl_release'])
    print('Assembly:', g.graph['genome_assembly'])
else:
    print('Skipping mouse tests: missing mus_musculus_externals_modified.yml')


Organism: mus_musculus
Release: 115
Assembly: 39


In [8]:
if HAS_MOUSE_YAML:
    example_id = pick_example_base_gene_id(api.track.graph)
    result = api.convert_identifier(example_id, from_release=SNAPSHOT_RELEASE, to_release=SNAPSHOT_RELEASE)
    display(result)
else:
    print('Skipping mouse smoke conversion (mouse graph not built).')


{'query_id': 'ENSMUSG00000000001',
 'graph_id': 'ENSMUSG00000000001',
 'match': 'exact',
 'known_versions': [],
 'from_release': 115,
 'from_release_inferred': False,
 'from_release_given': 115,
 'from_release_fallback': False,
 'from_databases': ['base_ensembl_gene'],
 'to_release': 115,
 'strategy': 'best',
 'final_database': 'ensembl_gene',
 'target_id': ['ENSMUSG00000000001.5'],
 'last_node': [['ENSMUSG00000000001.5', 'ENSMUSG00000000001.5']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': [],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

### 3C. Pig

In [9]:
pig_yaml = LOCAL_REPOSITORY / 'sus_scrofa_externals_modified.yml'
# A local file (Part 2) overrides the packaged default; either is fine.
shipped_pig_yaml = Path(idtrack.__file__).parent / 'default_config' / pig_yaml.name
HAS_PIG_YAML = pig_yaml.exists() or shipped_pig_yaml.exists()
if pig_yaml.exists():
    print('OK, local file:', pig_yaml)
else:
    print('OK, packaged default:' if HAS_PIG_YAML else 'MISSING (create it in Part 2 for pig):', shipped_pig_yaml.name)

OK, local file: idtrack_cache/sus_scrofa_externals_modified.yml


In [10]:
if HAS_PIG_YAML:
    organism, latest_release = api.resolve_organism('sus scrofa')
    SNAPSHOT_RELEASE = 115  # pinned (latest_release is Ensembl's newest)
    api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE)

    g = api.track.graph
    print('Organism:', g.graph['organism'])
    print('Release:', g.graph['ensembl_release'])
    print('Assembly:', g.graph['genome_assembly'])
else:
    print('Skipping pig tests: missing sus_scrofa_externals_modified.yml')


Organism: sus_scrofa
Release: 115
Assembly: 111


In [11]:
if HAS_PIG_YAML:
    example_id = pick_example_base_gene_id(api.track.graph)
    result = api.convert_identifier(example_id, from_release=SNAPSHOT_RELEASE, to_release=SNAPSHOT_RELEASE)
    display(result)
else:
    print('Skipping pig smoke conversion (pig graph not built).')


{'query_id': 'ENSSSCG00000000001',
 'graph_id': 'ENSSSCG00000000001',
 'match': 'exact',
 'known_versions': [],
 'from_release': 89,
 'from_release_inferred': True,
 'from_release_given': 115,
 'from_release_fallback': True,
 'from_databases': ['base_ensembl_gene'],
 'to_release': 115,
 'strategy': 'best',
 'final_database': 'ensembl_gene',
 'target_id': ['ENSSSCG00000027852.4'],
 'last_node': [['ENSSSCG00000027852.4', 'ENSSSCG00000027852.4']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': [],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

## 4. Optional: deeper integrity checks (advanced)

IDTrack exposes a developer-focused test harness called `TrackTests`.

If you are a power user and want extra confidence, you can load the graph in test mode and run invariants. Test mode needs the
tables the graph was built from (it reads Ensembl's tables through a database manager), and some tests are slow on large graphs.


In [12]:
# Example: build TrackTests for human and run one invariant
# (This is optional; skip if you just want to use IDTrack.)

api.build_graph(organism_name='homo_sapiens', snapshot_release=115, return_test=True)

tests = api.track  # TrackTests instance
# A relatively small check compared to full history stress-tests. Every check returns a result:
# `holds` says whether the invariant holds (it is also the truth value of the result), and
# `details` what was compared and every violation found.
result = tests.is_base_is_range_correct(verbose=False)
print(result.check, '-> holds:', result.holds)
print('gene ids compared:', result.details['compared'], '| differing:', result.details['n_differing'])


2026-11-12 23:11:59 INFO:idtrack._database_manager: Ensembl releases of homo_sapiens assembly 38 read from 'idtrack_cache/ensembl_https_index_homo_sapiens_assembly-38.json' (scanned up to release 116).


2026-11-12 23:11:59 INFO:idtrack._graph_maker: The graph is being read: idtrack_cache/graph_homo_sapiens_assembly38_min48_max115_narrow.npz


is_base_is_range_correct -> holds: True
gene ids compared: 127247 | differing: 0


## 5. If something fails

Common failure modes and fixes:

1. **Missing YAML**: every supported organism ships a default; for another organism run `02_prepare_new_external_yaml.ipynb`
   and create the `_modified.yml`. If a build warns that releases have no YAML entry, extend the file (Part 2, section 2.6).
2. **Permission errors**: change `IDTRACK_LOCAL_REPO` to a writable path.
3. **Network / MySQL issues**: REST/FTP access is required; direct MySQL access is optional (IDTrack falls back to HTTPS/FTP dumps;
   `IDTRACK_ENSEMBL_ACCESS=https` skips MySQL entirely).
4. **Disk full**: graphs can be large; move your local repository to a larger disk.
